In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [2]:
!pip3 install tslearn

In [3]:
import os
from os import listdir
import random
from typing import List
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import math
import sys
from glob import glob
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tslearn.datasets import UCR_UEA_datasets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler, TimeSeriesScalerMinMax
from typing import List
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy import stats
import keras
from keras import layers, models
from keras.layers import Dense, BatchNormalization, LSTM, Flatten, TimeDistributed, Dropout, GlobalMaxPooling2D, GlobalMaxPooling1D, MaxPooling2D, MaxPooling1D, Conv2D, Conv1D, Input, Reshape, ConvLSTM2D
from tslearn.preprocessing import TimeSeriesScalerMinMax
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy
from sklearn import metrics

## Load in Data

In [4]:
# import csv
path = "/content/gdrive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_insole_sensor_df.csv"

interpolated_insole_df = pd.read_csv(path)

In [5]:
path_imu = "/content/gdrive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/interpolated_alt_seg_imu_sensor_df.csv"

interpolated_imu_df = pd.read_csv(path_imu)

In [6]:
lower_body_imu_sensor_locations = ['Pelvis', 'RightUpperLeg', 'LeftUpperLeg', 'RightLowerLeg', 'LeftLowerLeg', 'RightFoot', 'LeftFoot']

In [7]:
# retain the sensors that are in the luo_imu_sensor_locations list
interpolated_imu_df = interpolated_imu_df[interpolated_imu_df['sensor_location'].isin(lower_body_imu_sensor_locations)]

In [8]:
interpolated_imu_df['gait_cycle_id'] = interpolated_imu_df['participant_id'].astype(str) + '_' + interpolated_imu_df['task'].astype(str) + '_' + interpolated_imu_df['segment_count'].astype(str)
interpolated_insole_df['gait_cycle_id'] = interpolated_insole_df['participant_id'].astype(str) + '_' + interpolated_insole_df['task'].astype(str) + '_' + interpolated_insole_df['segment_count'].astype(str)

In [9]:
# normalize each measurement column
interpolated_imu_df[['time', 'acceleration_x', 'acceleration_y', 'acceleration_z', 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']] = (interpolated_imu_df[['time', 'acceleration_x', 'acceleration_y', 'acceleration_z', 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']].apply(lambda x: (x - x.mean()) / x.std()) * 1000).astype(int)
interpolated_imu_df[['segment_count', 'participant_id', 'task', 'walk_mode']] = interpolated_imu_df[['segment_count', 'participant_id', 'task', 'walk_mode']].astype('int8')


In [10]:
interpolated_insole_df[['time', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative']] = (interpolated_insole_df[['time', 'Left_norm', 'Right_norm', 'Left_norm_cumulative', 'Right_norm_cumulative']] * 1000).astype(int)
interpolated_insole_df[['participant_id', 'segment_count']] = interpolated_insole_df[['participant_id', 'segment_count']].astype('int8')


In [11]:
le = LabelEncoder()
interpolated_insole_df['task'] = le.fit_transform(interpolated_insole_df['task'])
interpolated_insole_df['walk_mode'] = le.fit_transform(interpolated_insole_df['walk_mode'])

interpolated_imu_df['task'] = le.fit_transform(interpolated_imu_df['task'])
interpolated_imu_df['walk_mode'] = le.fit_transform(interpolated_imu_df['walk_mode'])

In [12]:
# Map groups and locations (same as before)
group_mapping = {
    'front': ['Hallux', 'Toes', 'Met1'],
    'middle': ['Met3', 'Met5', 'Arch'],
    'back': ['Heel_L', 'Heel_R', 'Arch']
}

location_mapping = {}
for group, sensors in group_mapping.items():
    location_mapping.update({sensor: f"Location_{i+1}" for i, sensor in enumerate(sensors)})

interpolated_insole_df['group'] = interpolated_insole_df['sensor_location'].map(
    lambda x: 'front' if x in group_mapping['front'] else
              'middle' if x in group_mapping['middle'] else
              'back' if x in group_mapping['back'] else None
)
# interpolated_insole_df['grouping'] = interpolated_insole_df['sensor_location'].map(location_mapping)


In [13]:
# take every row that has arch for a sensor location and duplicate it but now with the group being "back"
arch_rows = interpolated_insole_df[interpolated_insole_df['sensor_location'] == 'Arch']
back_rows = arch_rows.copy()
back_rows['group'] = 'back'
interpolated_insole_df = pd.concat([interpolated_insole_df, back_rows])

In [14]:
interpolated_insole_df.sort_values(by=['participant_id', 'task', 'time', 'sensor_location'], inplace=True)

In [15]:
# Group and aggregate data
grouped_df = (
    interpolated_insole_df
    .groupby(['group', 'gait_cycle_id', 'time', 'group'])[['Right_norm', 'Left_norm']]
    .sum()  # Aggregate (mean) the norm values
    .unstack(level=-1)  # Unstack the location to columns
)

In [16]:
# Reset index of grouped_df to flatten it for easy merging
grouped_df_reset = grouped_df.reset_index()
# Flatten multi-level columns into a single level
grouped_df_reset.columns = ['_'.join(map(str, col)) if isinstance(col, tuple) else col for col in grouped_df_reset.columns]
# rename gait_cycle_id_ in grouped_df_reset
grouped_df_reset = grouped_df_reset.rename({'gait_cycle_id_': 'gait_cycle_id', 'time_': 'time', 'group_': 'group'}, axis=1)


In [17]:
gc_insole_split = grouped_df_reset['gait_cycle_id'].str.split('_', expand=True)
grouped_df_reset['participant_id'] = gc_insole_split[0]
grouped_df_reset['segment_count'] = gc_insole_split[1]
grouped_df_reset['task'] = gc_insole_split[2]

In [18]:
grouped_df_reset.sort_values(by=['participant_id', 'task', 'time', 'group'], inplace=True)

In [19]:
interpolated_insole_df.sort_values(by=['participant_id', 'task', 'time', 'group'], inplace=True)

In [20]:
insole_dfs = pd.merge(interpolated_insole_df, grouped_df_reset, on=['gait_cycle_id', 'time', 'group'], how='left')

In [21]:
insole_dfs["Right_norm_grouped"] = insole_dfs["Right_norm_back"].fillna(0) + insole_dfs["Right_norm_middle"].fillna(0) + insole_dfs["Right_norm_front"].fillna(0).astype(int)
insole_dfs["Left_norm_grouped"] = insole_dfs["Left_norm_back"].fillna(0) + insole_dfs["Left_norm_middle"].fillna(0) + insole_dfs["Left_norm_front"].fillna(0).astype(int)

In [22]:
# drop right_norm_back, right_norm_middle...
insole_dfs = insole_dfs.drop(columns=['participant_id_y', 'segment_count_y', 'task_y', 'Right_norm_back', 'Right_norm_middle', 'Right_norm_front', 'Left_norm_back', 'Left_norm_middle', 'Left_norm_front'])

In [23]:
del interpolated_insole_df
del grouped_df_reset
del grouped_df

In [24]:
insole_dfs = insole_dfs.sort_values(by=['time', 'participant_id_x', 'sensor_location'])

In [25]:
# Filter sensor locations more efficiently
valid_sensors = {'Hallux', 'Heel_L', 'Heel_R', 'Met5'}
insole_dfs = insole_dfs[insole_dfs['sensor_location'].isin(valid_sensors)]


In [26]:
insole_dfs.columns

Index(['time', 'participant_id_x', 'task_x', 'Left_norm', 'Right_norm',
       'Left_norm_cumulative', 'Right_norm_cumulative', 'segment_count_x',
       'walk_mode', 'sensor_location', 'gait_cycle_id', 'group',
       'Right_norm_grouped', 'Left_norm_grouped'],
      dtype='object')

In [27]:
insole_dfs = insole_dfs.reset_index(drop=True)

In [28]:
insole_dfs.dtypes

,0
time,int64
participant_id_x,int8
task_x,int64
Left_norm,int64
Right_norm,int64
Left_norm_cumulative,int64
Right_norm_cumulative,int64
segment_count_x,int8
walk_mode,int64
sensor_location,object


In [29]:
insole_dfs.head()

,time,participant_id_x,task_x,Left_norm,Right_norm,Left_norm_cumulative,Right_norm_cumulative,segment_count_x,walk_mode,sensor_location,gait_cycle_id,group,Right_norm_grouped,Left_norm_grouped
0,0,1,0,0,968,838,3509,1,4,Hallux,1.0_0.0_1.0,front,2318.0,3.0
1,0,1,1,328,1,3786,460,1,4,Hallux,1.0_1.0_1.0,front,5.0,1512.0
2,0,1,2,879,0,4011,970,1,4,Hallux,1.0_2.0_1.0,front,1.0,2830.0
3,0,1,0,457,0,838,3509,1,4,Heel_L,1.0_0.0_1.0,back,162.0,830.0
4,0,1,1,0,177,3786,460,1,4,Heel_L,1.0_1.0_1.0,back,451.0,271.0


In [30]:
import pandas as pd

def melt_memory_friendly(df, id_vars, value_vars, var_name, value_name):
    def row_generator():
        for _, row in df.iterrows():
            row_dict = row[id_vars].to_dict()
            for var in value_vars:
                yield {**row_dict, var_name: var, value_name: row[var]}

    return pd.DataFrame.from_records(row_generator())

df_long = melt_memory_friendly(
    insole_dfs,
    id_vars=["time", "sensor_location", "gait_cycle_id", "walk_mode", "participant_id_x", "segment_count_x", "task_x", "group"],
    value_vars=["Left_norm", "Right_norm", "Left_norm_cumulative", "Right_norm_cumulative", "Left_norm_grouped", "Right_norm_grouped"],
    var_name="variable",
    value_name="value"
)


KeyboardInterrupt: 

In [ ]:
# Extract prefix (Right/Left) and base name in one step
df_long[['prefix', 'base_name']] = df_long['variable'].str.extract(r'^(Right|Left)_(.*)')

# Create grouped column
df_long["grouped_column"] = df_long["prefix"].str.lower() + "_" + df_long["group"]

In [ ]:
# Pivot data efficiently
df_transformed = df_long.pivot_table(
    index=["time", "sensor_location", "gait_cycle_id", "walk_mode", "participant_id_x", "segment_count_x", "task_x", "grouped_column"],
    columns="base_name",
    values="value"
).reset_index()

In [ ]:
# Rename columns
df_transformed = df_transformed.rename_axis(None, axis=1)
df_transformed.rename(columns={
    'grouped_column': 'sensor_location',
    'sensor_location': 'insole_location',
    'participant_id_x': 'participant_id',
    'task_x': 'task'
}, inplace=True)

In [ ]:
# Create foot side column
df_transformed['foot_side'] = df_transformed['sensor_location'].str.split('_').str[0]

# Filter out inconsistent foot-side and insole-location pairs
filtered_df = df_transformed[~(
    ((df_transformed['foot_side'] == 'right') & (df_transformed['insole_location'] == 'Heel_L')) |
    ((df_transformed['foot_side'] == 'left') & (df_transformed['insole_location'] == 'Heel_R'))
)]

In [ ]:
# Group data efficiently
side_groupings = filtered_df.groupby(
    ['participant_id', 'task', 'time', 'foot_side']
)[['norm', 'norm_grouped', 'norm_cumulative']].agg(list)

# Convert groupings to array using list comprehension
inputs_combo = [
    [sg[1][1], sg[2][1], sg[0][1], sg[1][0], sg[0][0], sg[0][2]]
    for sg in side_groupings.values
]

In [ ]:
# Convert to NumPy array
inputs_combo_array = np.array(inputs_combo)

# Drop duplicates directly
filtered_df.drop_duplicates(subset=['time', 'participant_id', 'task', 'foot_side'], inplace=True)

# Assign values efficiently
filtered_df[['acceleration_x', 'acceleration_y', 'acceleration_z',
             'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']] = inputs_combo_array


In [ ]:
# Drop unnecessary columns
filtered_df.drop(columns=['norm', 'norm_cumulative', 'norm_grouped', 'insole_location', 'sensor_location'], inplace=True)

# Rename foot_side to sensor_location
filtered_df.rename(columns={'foot_side': 'sensor_location'}, inplace=True)

In [ ]:
filtered_df.head()

In [ ]:
del df_long
del insole_dfs
del df_transformed

In [ ]:
filtered_df['time'] = filtered_df['time']/1000

In [ ]:
filtered_df.head()

In [ ]:
for col in ['acceleration_x', 'acceleration_y', 'acceleration_z', 'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']:
    filtered_df[col] = pd.to_numeric(filtered_df[col], downcast='integer', errors='coerce')

In [ ]:
import random

number_list = list(interpolated_imu_df['participant_id'].unique())
# Choose 3 numbers without replacement
test_three_numbers = random.sample(number_list, 2)
print(test_three_numbers)
train_list = [x for x in number_list if x not in test_three_numbers]

val_three_numbers = random.sample(train_list, 2)
print(val_three_numbers)

train_list = [x for x in train_list if x not in val_three_numbers]
print(train_list)

In [ ]:
train_imu = interpolated_imu_df.loc[interpolated_imu_df['participant_id'].isin(train_list)]
val_imu = interpolated_imu_df.loc[interpolated_imu_df['participant_id'].isin(val_three_numbers)]
test_imu = interpolated_imu_df.loc[interpolated_imu_df['participant_id'].isin(test_three_numbers)]

In [ ]:
train_insole = filtered_df.loc[filtered_df['participant_id'].isin(train_list)]
val_insole = filtered_df.loc[filtered_df['participant_id'].isin(val_three_numbers)]
test_insole = filtered_df.loc[filtered_df['participant_id'].isin(test_three_numbers)]

In [ ]:
train_insole.head()

In [ ]:
# concat the train with train, val with val, ...
train = pd.concat([train_imu, train_insole])
val = pd.concat([val_imu, val_insole])
test = pd.concat([test_imu, test_insole])

In [ ]:
train.sort_values(by=['participant_id', 'task', 'time', 'sensor_location'], inplace=True)

In [ ]:
del interpolated_imu_df
del df_transformed
del filtered_df

In [ ]:
# Optional: Reset index after merging
train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)


In [ ]:
train_gc = train.drop_duplicates(subset=['gait_cycle_id'], keep='first')

In [ ]:
# count how many unique gait cycle ids are in each walk mode
train_gc[['walk_mode']].value_counts()

In [ ]:
# Oversample smaller classes to balance the dataset
min_samples = int(train_gc['walk_mode'].value_counts().min())

In [ ]:
min_samples

In [ ]:
# Downsample to balance classes
balanced_gait_cycles_df = (
    train_gc.groupby('walk_mode')
    .apply(lambda group: group.sample(n=min_samples, replace=False, random_state=42))
    .reset_index(drop=True)
)


In [ ]:
balanced_gait_cycles_df.head()

In [ ]:
print(balanced_gait_cycles_df['walk_mode'].value_counts())

In [ ]:
filtered_interpolated_final_df = pd.merge(
    train,
    balanced_gait_cycles_df[['gait_cycle_id']],
    on='gait_cycle_id',
    how='inner'
)

In [ ]:
train.sort_values(by=['participant_id', 'task', 'time', 'sensor_location'], inplace=True)

In [ ]:
train.head(15)

In [ ]:
train = filtered_interpolated_final_df.copy()

In [ ]:
le = LabelEncoder()

train['gait_cycle_id'] = le.fit_transform(train['gait_cycle_id'])
train['sensor_location'] = le.fit_transform(train['sensor_location'])
train['task'] = le.fit_transform(train['task'])
train['walk_mode'] = le.fit_transform(train['walk_mode'])

val['gait_cycle_id'] = le.fit_transform(val['gait_cycle_id'])
val['sensor_location'] = le.fit_transform(val['sensor_location'])
val['task'] = le.fit_transform(val['task'])
val['walk_mode'] = le.fit_transform(val['walk_mode'])

test['gait_cycle_id'] = le.fit_transform(test['gait_cycle_id'])
test['sensor_location'] = le.fit_transform(test['sensor_location'])
test['task'] = le.fit_transform(test['task'])
test['walk_mode'] = le.fit_transform(test['walk_mode'])

In [ ]:
train['walk_mode'].value_counts()

In [ ]:
train['sensor_location'].value_counts()

In [ ]:
train = train.sort_values(by=['time', 'participant_id', 'sensor_location'])

In [ ]:
train.head(15)

In [ ]:
del filtered_interpolated_final_df
del train_gc
del balanced_gait_cycles_df
del train_insole
del train_imu

In [ ]:
def preprocess_data(df):
    # Initialize a list to hold data for each sensor location
    sensor_data_dict = {sensor: [] for sensor in df['sensor_location'].unique()}
    all_labels_y = []  # For storing labels for each gait cycle

    for gait_cycle_id, group_data in df.groupby('gait_cycle_id'):
        # Sort by time to ensure consistent order
        group_data = group_data.sort_values(by='time')

        # Process each sensor location
        for sensor_location, sensor_group in group_data.groupby('sensor_location'):
            # Extract the six IMU features
            sensor_array = sensor_group[['acceleration_x', 'acceleration_y', 'acceleration_z',
                                         'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z']].values

            # Pad or truncate to ensure length of 80
            if len(sensor_array) < 80:
                padding = np.zeros((80 - len(sensor_array), 6))  # Pad with 6 zeros
                sensor_array = np.vstack([sensor_array, padding])
            elif len(sensor_array) > 80:
                sensor_array = sensor_array[:80]

            # Append the processed sensor data for the current sensor location
            sensor_data_dict[sensor_location].append(sensor_array)

        # Extract walk mode labels for the gait cycle
        walk_mode_data = group_data['walk_mode'].values[:80]
        if len(walk_mode_data) < 80:
            walk_mode_data = np.pad(walk_mode_data, (0, 80 - len(walk_mode_data)), constant_values=-1)  # Use -1 for padding
        all_labels_y.append(walk_mode_data)

    # Convert sensor data for each sensor location into a NumPy array with shape [x, 80, 6]
    all_sensor_data = [np.array(sensor_data_dict[sensor]) for sensor in sorted(sensor_data_dict.keys())]

    # Convert labels to a NumPy array with shape [x, 80]
    all_labels_x = np.array(all_sensor_data)
    all_labels_y = np.array(all_labels_y)  # Shape: [x, 80]

    return all_sensor_data, all_labels_y


In [ ]:
train_all_data_x, train_all_labels_y = preprocess_data(train)

In [ ]:
val_all_data_x, val_all_labels_y = preprocess_data(val)

In [ ]:
test_all_data_x, test_all_labels_y = preprocess_data(test)

In [ ]:
train_all_data_x_np = np.array(train_all_data_x)
val_all_data_x_np = np.array(val_all_data_x)
test_all_data_x_np = np.array(test_all_data_x)

In [ ]:
train_all_labels_y = np.swapaxes(train_all_labels_y, 0, 1)
train_all_labels_y = train_all_labels_y[0]

val_all_labels_y = np.swapaxes(val_all_labels_y, 0, 1)
val_all_labels_y = val_all_labels_y[0]

test_all_labels_y = np.swapaxes(test_all_labels_y, 0, 1)
test_all_labels_y = test_all_labels_y[0]

In [ ]:
from keras.utils import to_categorical

# Assuming the labels are already numeric
train_all_labels_y = to_categorical(train_all_labels_y)
val_all_labels_y = to_categorical(val_all_labels_y)
test_all_labels_y = to_categorical(test_all_labels_y)

In [ ]:
train_all_labels_y.shape, train_all_labels_y[0]

In [ ]:
train_all_data_x_np.shape, train_all_labels_y.shape, val_all_data_x_np.shape, val_all_labels_y.shape, test_all_data_x_np.shape, test_all_labels_y.shape

In [ ]:
train_X = train_all_data_x_np
train_Y = train_all_labels_y

val_X = val_all_data_x_np
val_Y = val_all_labels_y

test_X = test_all_data_x_np
test_Y = test_all_labels_y

In [ ]:
n_obs, n_timesteps, n_features, n_classes = train_X.shape[1], train_X.shape[2], train_X.shape[3], train_Y.shape[1]

In [ ]:
n_obs, n_timesteps, n_features, n_classes

In [ ]:
train_X.shape

In [ ]:
train_X_list = [train_all_data_x_np[i] for i in range(train_all_data_x_np.shape[0])]
train_Y_list = [train_all_labels_y for i in range(train_all_data_x_np.shape[0])]

val_X_list = [val_all_data_x_np[i] for i in range(val_all_data_x_np.shape[0])]
val_Y_list = [val_all_labels_y for i in range(val_all_data_x_np.shape[0])]

test_X_list = [test_all_data_x_np[i] for i in range(test_all_data_x_np.shape[0])]

test_Y_list = [test_all_labels_y for i in range(test_all_data_x_np.shape[0])]

# Print the shape of each item in the list to verify
for idx, sensor_data in enumerate(train_X_list):
    print(f"Sensor {idx+1} shape: {sensor_data.shape}")
    print(f"Label {idx+1} shape: {train_Y_list[idx].shape}")
    print()

## hyperparameter tuning

In [ ]:
# !pip install scikeras

In [ ]:
# train_X.shape, train_Y.shape

In [ ]:
# val_X.shape, val_Y.shape

In [ ]:
# # Clears the session for TensorFlow
# tf.keras.backend.clear_session()

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.layers import Input, Conv1D, LSTM, Attention, Concatenate, Dense, Bidirectional, BatchNormalization, MaxPooling1D, Dropout, Flatten
# from sklearn.model_selection import GridSearchCV
# from scikeras.wrappers import KerasClassifier
# from tensorflow.keras.callbacks import EarlyStopping

# # Function to create the Keras model
# def create_model(input_shape=(80, 18), cnn_units=128, learning_rate=0.001, dropout=0.4, lstm_units=64, n_classes=5):

#     # Clears the session for TensorFlow
#     tf.keras.backend.clear_session()

#     inputs = Input(shape=input_shape)

#     # CNN layer
#     cnn_out = BatchNormalization()(inputs)
#     cnn_out = Conv1D(cnn_units, kernel_size=3, activation='relu')(cnn_out)
#     cnn_out = MaxPooling1D(pool_size=2)(cnn_out)
#     cnn_out = Dropout(dropout)(cnn_out)

#     # Bidirectional LSTM layer
#     lstm_out = Bidirectional(LSTM(units=lstm_units, dropout=0.4, recurrent_dropout=0.4, return_sequences=True))(cnn_out)

#     # Attention layer
#     attention = Attention()([cnn_out, lstm_out])

#     # Concatenate attention output with LSTM output
#     merged = Concatenate(axis=-1)([lstm_out, attention])

#     # MLP layer after attention and LSTM output
#     mlp_out = Flatten()(merged)
#     mlp_out = Dense(128, activation='relu')(mlp_out)
#     mlp_out = Dense(64, activation='relu')(mlp_out)
#     output_layer = Dense(n_classes, activation='softmax')(mlp_out)

#     # Model compilation
#     model = tf.keras.models.Model(inputs=inputs, outputs=output_layer)
#     optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
#     model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

#     return model


# # Wrap the model with KerasClassifier
# model = KerasClassifier(model=create_model, input_shape=(80, 18), verbose=2)

# # Define hyperparameter grid using double underscores for parameter names
# param_grid = {
#     'model__cnn_units': [64, 128, 256],  # Number of filters for the CNN layer
#     'model__learning_rate': [0.001, 0.0001, 0.00001],
#     'model__dropout': [0.3, 0.4, 0.5],
#     'model__lstm_units': [32, 64, 128],
#     'model__n_classes': [5]  # Number of classes for classification (fixed as per your data)
# }

# # Create the GridSearchCV object
# grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=None, scoring='accuracy')

# # Define the early stopping callback
# early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# # Fit the grid search with early stopping
# grid_result = grid.fit(train_X, train_Y, validation_data=(val_X, val_Y), callbacks=[early_stopping])

# # Print the best result
# print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


## Train Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, LSTM, Layer, Attention, Concatenate, Dense, Bidirectional, BatchNormalization, MaxPooling1D, Dropout, Flatten
from sklearn.model_selection import train_test_split

# Clears the session for TensorFlow
tf.keras.backend.clear_session()

# Define input shape and number of inputs
input_shape = (n_timesteps, n_features)
number_of_inputs = n_obs
number_of_sensors = len(train_X_list)

# Define 6 parallel branches with different inputs
branch_models = []
inputs = []
branches = []
attention_weights_list = []  # To store attention weights for each branch
count = 0

for _ in range(number_of_sensors):
    count += 1
    input_layer = Input(shape=input_shape)
    inputs.append(input_layer)

    # CNN layer
    cnn_out = BatchNormalization()(input_layer)
    cnn_out = Conv1D(filters=64, kernel_size=3, activation='relu')(cnn_out)
    cnn_out = MaxPooling1D(pool_size=2)(cnn_out)
    cnn_out = Dropout(0.45)(cnn_out)

    # Bidirectional LSTM layer
    lstm_out = Bidirectional(LSTM(units=128, dropout=0.4, recurrent_dropout=0.4, return_sequences=True))(cnn_out)

    # Project cnn_out to match the feature size of lstm_out
    cnn_out_projected = Dense(units=256)(cnn_out)

    # Attention layer
    attention = Attention()([cnn_out_projected, lstm_out])
    # Concatenate attention output with LSTM output
    merged = Concatenate(axis=-1)([lstm_out, attention])
    # Dense layer for individual branch output
    output_branch = Dense(units=n_classes, activation='softmax')(merged)

    # Add the branch model to the list
    branch_models.append(output_branch)
    branches.append(output_branch)

# Concatenate the outputs of the 6 branches and their corresponding attention weights
merged_output = Concatenate(axis=-1)(branches)
# merged_attention = Concatenate(axis=-1)(attention_weights_list)  # Get attention weights for each branch

# MLP layer
mlp_out = Flatten()(merged_output)
mlp_out = Dense(units=128, activation='relu')(mlp_out)
mlp_out = Dense(units=64, activation='relu')(mlp_out)
output_layer = Dense(units=n_classes, activation='softmax')(mlp_out)

# Model outputs both classification and attention weights
model = tf.keras.models.Model(inputs=inputs, outputs=[output_layer])

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

# Compile the model
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Assuming each entry in train_X_split_trimmed corresponds to one sensor's data
# Make sure that train_X_split_trimmed has exactly 6 entries for the 6 sensors
history = model.fit(train_X_list, train_Y_list, epochs=200, batch_size=128, callbacks=[callback], validation_data=(val_X_list, val_Y_list))



In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_X_list, test_Y_list)

In [ ]:
# Check shapes of test data before evaluation
print("Test X shape:", test_X_list[0].shape)
print("Test Y shape:", test_Y_list[0].shape)

In [ ]:
#predictions
predictions = model.predict(test_X_list)

In [ ]:
# Convert predictions to class labels
predicted_classes = tf.argmax(predictions, axis=-1)

# Convert true labels to class labels (if needed)
true_classes = tf.argmax(test_Y_list, axis=-1)

# Calculate accuracy
accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted_classes, true_classes), tf.float32))
print(f"Test accuracy: {accuracy.numpy() * 100:.2f}%")

# Display predicted and true classes for further inspection
print("Predicted classes:", predicted_classes.numpy())
print("True classes:", true_classes.numpy())

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score

# Convert predictions and true labels to NumPy arrays
# The variables are already numpy arrays, so no conversion is needed.
predicted_classes_np = np.array(predicted_classes)
true_classes_np = np.array(true_classes[0])

# Generate a classification report (includes precision, recall, F1-score for each class)
report = classification_report(true_classes_np, predicted_classes_np, digits=4)
print("Classification Report:\n", report)

# Compute confusion matrix
conf_matrix = confusion_matrix(true_classes_np, predicted_classes_np)
print("Confusion Matrix:\n", conf_matrix)

# Calculate F1-score and sensitivity (recall) per class
f1_scores = recall_score(true_classes_np, predicted_classes_np, average=None)
print(f"F1-Scores (per class): {f1_scores}")

# Custom calculation for specificity per class
specificities = []
for i in range(conf_matrix.shape[0]):
    tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
    fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificities.append(specificity)

print(f"Specificities (per class): {specificities}")

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score # Import the f1_score function
from sklearn.metrics import precision_recall_fscore_support

# Calculate F1-score and sensitivity (recall) per class
f1_scores = recall_score(true_classes_np, predicted_classes_np, average=None)
print(f"F1-Scores (per class): {f1_scores}")
f1 = f1_score(true_classes_np, predicted_classes_np, average='weighted')
# overall F1 score average
overall_F1 = round(math.fsum(f1_scores) / len(f1_scores),2)
print(f"Overall F1 Score: {overall_F1}")


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
res = []
for l in [0,1,2,3,4]:
     prec,recall,_,_ = precision_recall_fscore_support(np.array(true_classes_np)==l,
                                                  np.array(predicted_classes_np)==l,
                                                  pos_label=True,average=None)
     res.append([l,recall[0],recall[1]])

sens_spes = pd.DataFrame(res,columns = ['class','specificity','sensitivity'])
print(sens_spes)

overall_specificity = sens_spes['specificity'].mean()
overall_sensitivity = sens_spes['sensitivity'].mean()
print(f"Overall specificity: {round(overall_specificity,2)}")
print(f"Overall sensitivity: {round(overall_sensitivity, 2)}")

## Confusion Matrix

In [ ]:
true_classes = true_classes[0].numpy()
predicted_classes = predicted_classes.numpy()

In [ ]:
# Compute the confusion matrix
result = confusion_matrix(true_classes, predicted_classes)

In [ ]:
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = result, display_labels = ['slope down', 'slope up', 'stairs down', 'stairs up', 'walk'])

cm_display.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
# Create the confusion matrix display using the from_predictions method
cm_display = ConfusionMatrixDisplay.from_predictions(
    true_classes,
    predicted_classes,
    display_labels=['slope down', 'slope up', 'stairs down', 'stairs up', 'walk'],
    normalize='true'  # Normalize by the true labels
)

cm_display.plot(cmap=plt.cm.Blues)

plt.title("Normalized Confusion Matrix")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

In [ ]:
# 0 GR, 1 BNKR, 2 BNKL, 3 SLPD, 4 SLPU, 5 STD, 6 STU, 7 CS, 8 FE

In [ ]:
# 0 GR, 1 SLPD, 2 SLPU, 3 STD, 4 STU, 5 CS, 6 FE